# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 5: Retrieval-Augmented Generation</font>

# <font color="#003660">RAG Basics</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know the basics of Retrieval-Augmented Generation (RAG), <br>
        ... can implement a RAG chain from scratch.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [Nguyen (2024): Code a simple RAG from scratch](https://huggingface.co/blog/ngxson/make-your-own-rag)

In [ ]:
%pip install -q pymupdf4llm pymupdf_layout langchain-openai langchain-huggingface sentence-transformers python-dotenv tqdm

## Where We Are

* **5.01:** the whole document goes into the prompt. Fine for a short document, too expensive for a long one.
* **Now:** send only the pieces that matter for the question. We build this from scratch, step by step.

## Retrieval-Augmented Generation

**In short: first find the relevant pieces, then let the LLM answer from them.**

![RAG](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/imgs/RAG.png)

(Image adapted from [Kaltenpoth and Müller (2024)](https://energy.acm.org/eir/dont-touch-the-power-line-a-proof-of-concept-for-aligned-llm-based-assistance-systems-to-support-the-maintenance-in-the-electricity-distribution-system/))

* The **retriever** (usually an embedding model) finds the k document pieces that are most similar to the question.
* These pieces are the **context**.
* The **generator**, an LLM, answers the question from this context.

RAG was introduced by [Lewis et al. (2020)](https://proceedings.neurips.cc/paper/2020/hash/6b493230205f780e1bc26945df7481e5-Abstract.html).

## Setup

In [ ]:
import os
import urllib.request
import pymupdf4llm
from tqdm.notebook import tqdm
from IPython.display import display, Markdown

from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

### Choose where the model runs

We use two models:

* a **chat model** that writes the answers (the generator). It runs where you choose below.
* an **embedding model** that turns texts into vectors (the retriever). It always runs on your computer, see below.

| `provider` | Where the model runs | `model_name` | Good to know |
|---|---|---|---|
| `"ollama"` | **Ollama** on your computer | `qwen3.5:9b`, `qwen3.5:4b`, `qwen3.5:2b` | free, no account; install [Ollama](https://ollama.com/download) first, then download the model once in a terminal: `ollama pull qwen3.5:9b` (6.6 GB; 4b: 3.3 GB; 2b: 2.7 GB); without a GPU, it runs on the CPU, which is slower |
| `"colab"` | **Ollama on Google Colab** | the same three | free, needs a Google account; the cell installs Ollama in the Colab runtime and downloads the model, as in 4.01; choose a GPU runtime first (Runtime > Change runtime type > T4 GPU) |
| `"huggingface"` | **Hugging Face endpoints** of the course | the same three, one endpoint each | key and addresses from the course; if an endpoint was asleep, the first request takes 1 to 2 minutes |
| `"ai-gateway"` | **AI-Gateway** of Paderborn University | `gwdg.qwen3.8-27b-no-think` (thinking off), `gwdg.qwen3.8-27b` (thinking on) | your university key (see [setup/SETUP_AI_GATEWAY.md](https://github.com/olivermueller/amlta-2026/blob/main/setup/SETUP_AI_GATEWAY.md)); only in the university network or the VPN, not on Colab; about 10 requests per minute |
| `"other"` | **another** OpenAI-compatible API | the name your provider uses | base URL and key in `.env` (`OTHER_BASE`, `OTHER_KEY`) |

Keys and addresses come from the `.env` file in the repo folder (see `.env.example`). Without a `.env` file, for example on Colab, the cell asks for them.

In [ ]:
# Run this cell: it only defines setup_model(). We call it before every ChatOpenAI, with the provider and the model.
import getpass
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

from dotenv import load_dotenv

load_dotenv()  # reads the .env file if there is one: this folder first, then the folders above (the repo folder)
IN_COLAB = "google.colab" in sys.modules

PROVIDERS = ["ollama", "colab", "huggingface", "ai-gateway", "other"]
QWEN = {  # model_name: (download size in Ollama, the name of its endpoint address in .env, the model on Hugging Face)
    "qwen3.5:9b": ("6.6 GB", "HF_GATEWAY_BASE_9B", "unsloth/Qwen3.5-9B-GGUF"),
    "qwen3.5:4b": ("3.3 GB", "HF_GATEWAY_BASE_4B", "unsloth/Qwen3.5-4B-GGUF"),
    "qwen3.5:2b": ("2.7 GB", "HF_GATEWAY_BASE_2B", "unsloth/Qwen3.5-2B-GGUF"),
}
GATEWAY_QWEN = {  # on the AI-Gateway, the model name decides about thinking
    "gwdg.qwen3.8-27b-no-think": "thinking off",
    "gwdg.qwen3.8-27b": "thinking on",
}


def value(env_name, question, secret=False):
    """Takes the value from .env, or asks for it (keys are typed hidden)."""
    found = os.getenv(env_name)
    if found:
        return found.strip()
    answer = getpass.getpass(question).strip() if secret else input(question).strip()
    os.environ[env_name] = answer  # asked once: the next call of setup_model() takes it from here
    return answer


def start_ollama(model, size, colab=False):
    """Makes sure that the Ollama server runs and has the model. Downloads the model only on Colab."""
    url = (os.getenv("OLLAMA_BASE") or "http://localhost:11434").rstrip("/")

    def running():
        try:
            urllib.request.urlopen(url, timeout=2)
            return True
        except OSError:
            return False

    if colab and shutil.which("ollama") is None:
        # the same steps as in 4.01: the installer needs pciutils and zstd
        print("Installing Ollama in the Colab runtime (about a minute) ...")
        subprocess.run("sudo apt-get install -y -qq pciutils zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null",
                       shell=True, check=True)
    if not running():
        if shutil.which("ollama") is None:
            raise RuntimeError("Ollama is not installed. Install it from https://ollama.com/download and run this cell again.")
        # on Colab, the installer may say that the server is running, but it is not; so we start it ourselves (as in 4.01)
        print("Starting the Ollama server ...")
        ollama_log = open("ollama.log", "w")  # a pipe that nobody reads could fill up and block the server
        subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
        for _ in range(30):
            if running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("The Ollama server did not start. Look into ollama.log.")

    installed = [m["name"] for m in json.load(urllib.request.urlopen(url + "/api/tags"))["models"]]
    if model not in installed and not colab:  # on your computer, you download the model yourself, once
        raise RuntimeError(f"{model} is not downloaded yet. Run this command in a terminal (it downloads {size}), "
                           f"then run this cell again:\n\n    ollama pull {model}\n")
    if model not in installed:
        print(f"WARNING: {model} is not here yet. It is downloaded now ({size}), this can take several minutes.")
        request = urllib.request.Request(url + "/api/pull", data=json.dumps({"model": model}).encode())
        shown = -1
        for line in urllib.request.urlopen(request):
            status = json.loads(line)
            if "error" in status:
                raise RuntimeError(status["error"])
            if status.get("total"):
                percent = int(100 * status.get("completed", 0) / status["total"]) // 10 * 10
                if percent != shown:
                    print(f"  {percent} %", end="  ")
                    shown = percent
        print("\nDownload finished.")
    return url + "/v1/"


def wait_until_awake(base_url, api_key):
    """A sleeping Hugging Face endpoint answers 503 until it has started; we wait for it."""
    request = urllib.request.Request(base_url + "/models", headers={"Authorization": f"Bearer {api_key}"})
    for attempt in range(40):  # up to about 6 minutes
        try:
            urllib.request.urlopen(request, timeout=10)
            return
        except urllib.error.HTTPError as error:
            if error.code == 403:
                raise RuntimeError("The endpoint answers 403: it is paused (the course endpoints only run during the seminar), "
                                   "or your key has no access to it. Choose another provider, or check HF_GATEWAY_KEY and the address in .env.")
            if error.code != 503:
                raise RuntimeError(f"The endpoint answers {error.code}. Check the key and the address.")
        except OSError:
            pass
        if attempt == 0:
            print("The endpoint is asleep and starts now. This takes a few minutes ...")
        time.sleep(8)
    raise RuntimeError("The endpoint did not wake up. Try again in a few minutes.")


def setup_model(model):
    """Checks {"model_name": ..., "provider": ...}, sets the address and the key, and prints what to keep in mind.
    Returns base_url, api_key and the model name for ChatOpenAI."""
    name = model.get("model_name", "").strip()
    provider = model.get("provider", "").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown provider {provider!r}. Choose one of: {', '.join(PROVIDERS)}.")
    if provider in ("ollama", "colab", "huggingface") and name not in QWEN:
        raise ValueError(f"The provider {provider!r} has these models: {', '.join(QWEN)}. Not {name!r}.")
    notes = []

    if provider in ("ollama", "colab"):
        if provider == "ollama" and IN_COLAB:
            raise ValueError("This notebook runs on Google Colab: use the provider 'colab'.")
        if provider == "colab" and not IN_COLAB:
            raise ValueError("The provider 'colab' only works on Google Colab. On your own computer, use 'ollama'.")
        if provider == "colab" and subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
            notes.append("This Colab runtime has no GPU: answers take long. Better: Runtime > Change runtime type > T4 GPU, then run the notebook again.")
        if provider == "ollama":
            notes.append("The model runs on this computer. Without a GPU, it runs on the CPU, and answers take longer.")
        base_url, api_key = start_ollama(name, QWEN[name][0], colab=(provider == "colab")), "ollama"  # Ollama needs no key
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): Ollama returns only one.")

    elif provider == "huggingface":
        _, env_name, name = QWEN[name]  # name: the model on Hugging Face
        base_url = value(env_name, f"Paste the address of the Hugging Face endpoint for {model['model_name']}: ").rstrip("/")
        if "<" in base_url:
            raise ValueError(f"The address for {model['model_name']} is still a placeholder in .env ({env_name}). "
                             "Paste the address from the course there, or choose another model or provider.")
        if not base_url.endswith("/v1"):
            base_url += "/v1"  # the OpenAI-compatible API of the endpoint
        api_key = value("HF_GATEWAY_KEY", "Paste your Hugging Face key (for a public endpoint, just press Enter): ", secret=True)
        api_key = api_key or "none"  # a public endpoint needs no key, but the client needs some text
        wait_until_awake(base_url, api_key)
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): the endpoint refuses them (HTTP 400).")

    elif provider == "ai-gateway":
        if IN_COLAB:
            notes.append("Colab runs outside the university network and usually cannot reach the AI-Gateway.")
        base_url = os.getenv("AI_GATEWAY_BASE") or "https://ai-gateway.uni-paderborn.de/v1"
        api_key = value("AI_GATEWAY_KEY", "Paste your AI-Gateway key: ", secret=True)
        if name not in GATEWAY_QWEN:
            notes.append(f"{name!r} is not one of the Qwen models of the course ({', '.join(GATEWAY_QWEN)}). "
                         "llm.root_client.models.list() lists the models of your account.")
        notes.append("Thinking cannot be switched with reasoning_effort here (the gateway ignores it): the model name decides. "
                     "gwdg.qwen3.8-27b-no-think = off, gwdg.qwen3.8-27b = on.")
        notes.append("Only in the university network or the VPN. About 10 requests per minute: if a cell pauses, the client waits and continues.")

    else:  # another OpenAI-compatible API
        if not name:
            raise ValueError("Give the model name of your provider as model_name.")
        base_url = value("OTHER_BASE", "Base URL of the API (e.g. https://api.openai.com/v1/): ")
        api_key = value("OTHER_KEY", "Your API key: ", secret=True)
        notes.append("Check in the documentation of your provider how thinking is switched; "
                     "the extra_body of ChatOpenAI may need other fields than reasoning_effort.")

    print(f"Provider: {provider} | model: {name}")  # the key is never printed
    print("Keep in mind:")
    for note in notes:
        print("  -", note)
    return base_url, api_key, name

**Before every `ChatOpenAI`, we call `setup_model()`** with the provider and the model, for example `setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})`. It checks whether the combination exists, sets the address and the key, and prints what to keep in mind. It returns three values, which we split directly into `BASE_URL`, `API_KEY`, and `MODEL_NAME`. To switch the model, change the dict there.

### The embedding model runs on your computer

With every provider above, the **embedding model** runs on your computer: [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), a small model that turns texts into vectors.

**Warning:** the first run downloads it from Hugging Face (about 90 MB). Without a GPU, it runs on the CPU, which is slower.

MiniLM reads at most **256 tokens** per text and silently cuts off the rest. So our chunks must be short.

In [ ]:
from huggingface_hub import try_to_load_from_cache

RETRIEVER_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # embedding model: turns texts into vectors, runs on your computer
if not isinstance(try_to_load_from_cache(RETRIEVER_NAME, "config.json"), str):
    print(f"WARNING: {RETRIEVER_NAME} is downloaded on first use (about 90 MB). Without a GPU, it runs on the CPU, which is slower.")

In [ ]:
# downloads the documents, if they are not here yet
REPO_URL = "https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/"
for path in [
    "documents/Game_of_Thrones.pdf",
    "documents/How_I_Met_Your_Mother.pdf",
    "markdown_documents/Game_of_Thrones.md",
    "markdown_documents/How_I_Met_Your_Mother.md",
]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO_URL + path, path)

## Part 1: From PDF to Markdown

**In short: LLMs need text, so we convert the PDFs to Markdown.**

* We use two Wikipedia articles: *Game of Thrones* and *How I Met Your Mother*. With two topics, you can see that retrieval finds the right article and does not just get lucky.
* [PyMuPDF4LLM](https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/) converts PDFs to Markdown for LLMs.

In [ ]:
documents_path = "documents"
markdown_documents_path = "markdown_documents"

In [ ]:
documents = sorted(os.listdir(documents_path))

for document in tqdm(documents):
    document_path = os.path.join(documents_path, document)
    md_file = pymupdf4llm.to_markdown(document_path)
    md_file_path = os.path.join(markdown_documents_path, document.replace(".pdf", ".md"))
    with open(md_file_path, "w", encoding="utf-8") as file:
        file.write(md_file)

## Part 2: Indexing

**In short: cut the documents into chunks, turn every chunk into a vector, and store the vectors.**

![Vector database](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/imgs/vectordb.png)

(Adapted from [Xie et al. (2023)](https://doi.org/10.1109/BigDIA60676.2023.10429609))

A vector database has two steps:

| Step | When | What happens |
|---|---|---|
| **Indexing** | once | the embedding model turns every chunk into a vector |
| **Querying** | for every question | a similarity search, for example with cosine similarity |

This is the indexing process that we build now:

![](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_indexing.png?w=1650&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=4b9e544a7a3ec168651558bce854eb60)

(Source: [Langchain Docs](https://docs.langchain.com/oss/python/langchain/rag#faiss))

### Loading Documents

In [ ]:
markdown_documents = sorted(os.listdir(markdown_documents_path))

md_files = []

for markdown_document in markdown_documents:
    markdown_document_path = os.path.join(markdown_documents_path, markdown_document)
    with open(markdown_document_path, encoding="utf-8") as file:
        md_files.append([markdown_document, file.read()])

In [ ]:
display(Markdown(md_files[0][1][:1000]))

In [ ]:
print(md_files[0][1][:1000])

### Chunking Texts

Whole documents are too long for one embedding and too long for a prompt. So we cut them into **chunks**.

We cut by characters: in English, about 4 characters are one token, Our embedding model reads at most 256 tokens, and Wikipedia articles have many names and numbers, which need more tokens. So we use 600 characters, roughly 150 to 250 tokens. Neighbouring chunks share some text (the **overlap**), so that a sentence at the border is not torn apart ([Wang et al., 2024](https://doi.org/10.18653/v1/2024.emnlp-main.981)).

In [ ]:
for md_file in md_files:
    print("File name:", md_file[0], "| characters:", len(md_file[1]), "| about", len(md_file[1]) // 4, "tokens")

In [ ]:
OVERLAP = 60        # characters shared by two neighbouring chunks
CHUNK_LENGTH = 600  # characters per chunk, roughly 150 to 250 tokens (the embedding model reads at most 256)

md_files_chunked = []
for md_file in md_files:
    text = md_file[1]
    for start in range(0, len(text), CHUNK_LENGTH - OVERLAP):
        md_files_chunked.append([md_file[0], text[start:start + CHUNK_LENGTH]])

print(len(md_files_chunked), "chunks")

In [ ]:
print(md_files_chunked[0])
print(md_files_chunked[1])

### Loading the Embedding Model

The embedding model runs on your computer. LangChain's `HuggingFaceEmbeddings` loads it from Hugging Face with the `sentence-transformers` package. It has the same methods as every embedding model in LangChain: `embed_documents()` for many texts and `embed_query()` for one question.

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name=RETRIEVER_NAME)

In [ ]:
embeddings = embedding_model.embed_documents([md_files_chunked[0][1]])

In [ ]:
print(list(embeddings[0][:10]) + ["..."], len(embeddings[0]))

### The vector database

We use the easiest (and slowest) way to store the vectors: a plain Python list.

We embed **all** chunks in one call. On a CPU, this takes a while, so run it only once.

In [ ]:
all_embeddings = embedding_model.embed_documents([chunk for source, chunk in md_files_chunked])

VECTOR_DB = []
for (source, chunk), embedding in zip(md_files_chunked, all_embeddings):
    VECTOR_DB.append({"embeddings": embedding, "content": chunk, "metadata": {"source": source}})

print(len(VECTOR_DB), "chunks in the vector database")

In [ ]:
def cosine_similarity(a, b):
    dot_product = sum([x * y for x, y in zip(a, b)])
    norm_a = sum([x ** 2 for x in a]) ** 0.5
    norm_b = sum([x ** 2 for x in b]) ** 0.5
    return dot_product / (norm_a * norm_b)

## Part 3: Retrieval

**In short: embed the question, and return the chunks that are most similar to it.**

### Your turn

Complete `retrieve()`: compute the cosine similarity between the question and every chunk in `VECTOR_DB`, and add a `[chunk, similarity]` pair to `similarities` for each one.

In [ ]:
def retrieve(query, top_n=3):
    query_embedding = embedding_model.embed_query(query)
    # temporary list to store (chunk, similarity) pairs
    similarities = []
    # TODO: calculate cosine similarity between query and each chunk in the VECTOR_DB
    # Hint: each VECTOR_DB entry is a dictionary with keys "embeddings" (embeddings) and "content" (text chunk)



    # write your code here




    # sort by similarity in descending order, because higher similarity means more relevant chunks
    similarities.sort(key=lambda x: x[1], reverse=True)
    # finally, return the top N most relevant chunks
    return similarities[:top_n]

In [ ]:
retrieved_docs = retrieve("Who plays Jon Snow?")

for doc in retrieved_docs:
    print(doc)

## Part 4: Generation

**In short: the LLM answers the question from the retrieved chunks only.**

The generator is the chat model you chose in the setup, with `ChatOpenAI` from 5.01.

In [ ]:
BASE_URL, API_KEY, MODEL_NAME = setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})  # provider and model: see the table in the setup
llm = ChatOpenAI(
    model=MODEL_NAME,  # on the AI-Gateway, the model decides about thinking: gwdg.qwen3.8-27b-no-think (off) or gwdg.qwen3.8-27b (on)
    base_url=BASE_URL,
    api_key=API_KEY,
    temperature=0,
    extra_body={
        "max_tokens": 512,  # the token limit (here, because Ollama ignores max_completion_tokens, the name that ChatOpenAI uses)
        "reasoning_effort": "none",  # thinking off; "low", "medium", or "high" switches it on (Ollama, Hugging Face endpoint)
    },
)

### The generation prompt

In [ ]:
prompt = '''Use only the following context chunks to answer the question: {input_query}
Don't make up any new information. Here are the chunks:
{chunks}
Now answer the question: {input_query}'''

### Retrieve, then generate

Now let's bring everything together. `generate()` sends a list of messages to the generator and returns the text of the answer.

In [ ]:
def generate(messages):
    return llm.invoke(messages).content

#### Step 1: Retrieve

In [ ]:
input_query = "Who plays Jon Snow?"
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
    print(f' - (similarity: {similarity:.2f}) {chunk}')

#### Step 2: Generate

In [ ]:
chunks = '\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])
question = prompt.format(input_query=input_query, chunks=chunks)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": question}
]
response = generate(messages)

In [ ]:
print(response)

## Part 5: The RAG Chain

**In short: retrieval and generation in one function.**

### Your turn

Complete `generate_rag_response()`: retrieve the chunks, print them, fill the prompt, generate the answer, and return it. You find every piece in the cells above.

In [ ]:
def generate_rag_response(input_query):
    # TODO: retrieve relevant chunks from the VECTOR_DB
    # TODO: print chunks
    # TODO: create a prompt using the retrieved chunks and the input query
    # TODO: generate a response using the RAG model
    # TODO: return the generated response


    # write your code here



    pass

### Try it

Ask your own question about one of the two series.

In [ ]:
input_query = input('Ask me a question: ')
print(generate_rag_response(input_query))

<a href="https://imgflip.com/i/9fsgxc"><img src="https://i.imgflip.com/9fsgxc.jpg" title="made at imgflip.com"/></a><div><a href="https://imgflip.com/memegenerator">from Imgflip Meme Generator</a></div>

## What You Learned

* **RAG** = first retrieve the relevant chunks, then generate the answer from them.
* **Indexing** (once): PDF → Markdown → chunks → embeddings → vector database.
* **Querying** (for every question): embed the question and find the most similar chunks with cosine similarity.
* The generator sees only the retrieved chunks, not the whole document. That saves tokens.

Next: **5.03 RAG in 10 Lines of Code**. The same with LangChain.